In [54]:
import polars as pl
import torch
from torch import nn

from typing_extensions import Self

In [55]:
torch.cuda.is_available()

True

## Masked softmax

In [56]:
class Softmax(nn.Module):
    def __init__(self: Self, dim: float, temperature: float = 1.) -> None:
        super().__init__()
        if temperature == 0:
            raise ValueError('Temperature cannot be zero.')
        self.temperature = temperature
        self.dim = dim
    def forward(self: Self, X: torch.Tensor) -> torch.Tensor:
        # subtract the max for stability with large inputs (thanks, gemini). add eps so there's never division by zero (no longer necessary now that we subtract?).
        intermediate = torch.tensor(torch.e).pow( (X - torch.max(X, dim=self.dim, keepdim=True)[0]) .div(self.temperature)) # + 1e-10
        # unsqueeze to align dimensions (on dim zero want something like (2, 4, 1) rather than just (2, 4))
        denominator = torch.unsqueeze(intermediate.sum(self.dim), self.dim)
        return intermediate.div(denominator)

In [57]:
a = torch.tensor([ # batch size 2, 3 queries, len 2
    [ 
        [0., 0.], # q1
        [1., 1.], # q2
        [20., 20.] # q3
    ],
    [ 
        [100., 100.], # q1
        [1000., 1000.], # q2
        [1., 0.] # q3
    ],
])

In [58]:
dim = 0
softmax = Softmax(dim = dim)
softmax(a) - torch.nn.functional.softmax(a, dim = dim)

tensor([[[0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00],
         [3.1086e-15, 1.1102e-15]]])

In [59]:
class MaskedSoftmax(nn.Module):
    def __init__(self, dim: int, temperature: float = 1.):
        super().__init__()
        self.softmax = Softmax(dim, temperature)
    def forward(self, X: torch.Tensor, mask: torch.Tensor):
        """
        
        :param X: Input to be softmaxed rowwise.
        :param mask: Tensor of booleans with same shape as input. True masks the value off.
        """
        M = X.masked_fill(mask, -torch.inf)
        weights = self.softmax(M)
        # if a row is all masked, weights will contain nans
        return torch.nan_to_num(weights, 0)

Success!

In [60]:
raw_weights_test = torch.tensor(
    [
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
    ]
)

raw_weights_mask_test = torch.tensor(
    [
        [False, True, True],
        [False, False, True],
        [False, False, False]
    ]
)

masked_softmax = MaskedSoftmax(-1)

masked_softmax(raw_weights_test, raw_weights_mask_test)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

## Attention

In [61]:
class Attention(nn.Module):
    def __init__(self, dropout = 0.5):
        """Scaled dot product attention with masked softmax.

        :param num_hidden: Number of neurons in the hidden layer.
        :param max_length: Maximum length of a sequence.
        :param dropout: Hyperparameter for dropout.
        """
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.softmax = MaskedSoftmax(-1)
    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        
        :param Q: Query tensor of size (n in batch, number of queries, query length).
        :param K: Key tensor (n in batch, number of keys, query length).
        :param V: Value tensor (n in batch, number of keys, value length).
        :param attention_mask: Boolean mask for masked softmax (n in batch, number of queries, number of keys).
        """
        sqrt_d = torch.tensor(Q.shape[-1]) ** 0.5
        raw_weights = torch.matmul(Q, torch.transpose(K, -2, -1)).div(sqrt_d)
        attention_weights = self.dropout(self.softmax(raw_weights, attention_mask)) # why dropout here rather than somewhere else?
        return torch.matmul(attention_weights, V)

In [62]:
queries_test= torch.tensor(
    [
        [1., 1., 0.], # query 1
        [1., 2., 3.], # query 2
        [0., 0., 0.], # query 3 (do not attend to)
    ]
)

keys_test = torch.tensor(
    [
        [2., 0.1, 0.2], # key 1 
        [2., 0.7, 3.], # key 2
    ]
)

values_test = torch.tensor(
    [
        [1., 1., 1., 0.5], # value 1 
        [1., 2., 3., 0.8], # value 2
    ]
)

raw_weights_mask_test = torch.tensor(
    [
        [False, False,],
        [True, False,],
        [True, True,] # do not attend to q3
    ]
)

In [63]:
attention = Attention(dropout=0)
attention(queries_test, keys_test, values_test, raw_weights_mask_test)

tensor([[1.0000, 1.5857, 2.1715, 0.6757],
        [1.0000, 2.0000, 3.0000, 0.8000],
        [0.0000, 0.0000, 0.0000, 0.0000]])

## Multi-head attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, input_hidden: int, num_hidden: int, num_heads: int, dropout: float = 0.) -> None:
        """Initializes multi-head attention.
        
        :param input_hidden: Dimensionality of the input vectors. 
        :param num_hidden: Number of neurons in the hidden layer per head.
        :param num_heads: Number of heads to concatenate.
        :param dropout: Attention dropout hyperparameter.
        """
        super().__init__()
        self.input_hidden = input_hidden
        self.num_heads = num_heads
        self.per_hidden = num_hidden
        self.total_hidden = num_hidden*num_heads
        self.Q_w = nn.LazyLinear(self.total_hidden, bias = False)
        self.K_w = nn.LazyLinear(self.total_hidden, bias = False)
        self.V_w = nn.LazyLinear(self.total_hidden, bias = False)
        self.C_w = nn.LazyLinear(self.input_hidden, bias = False) # restores to input dimensionality
        self.attention = Attention(dropout)

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, attention_mask: torch.Tensor | bool = False) -> torch.Tensor:
        """Forward pass of MHA.
        
        :param Q: Query tensor of size (n in batch, number of queries, query embedding dimension).
        :param K: Key tensor (n in batch, number of keys, key embedding dimension).
        :param V: Value tensor (n in batch, number of keys, value embedding dimension).
        :param attention_mask: Boolean mask for masked softmax (n in batch, number of queries, number of keys).
        """
        if attention_mask is True:
            attention_mask = torch.tril(torch.ones(Q.shape[0], Q.shape[1], K.shape[1], device=Q.device, dtype=torch.bool))
        elif attention_mask is False:
            attention_mask = torch.zeros(Q.shape[0], Q.shape[1], K.shape[1], device=Q.device, dtype=torch.bool)
        q = self.reorder_tensor(self.Q_w(Q)) # (batch size*heads, queries, hidden per head)
        k = self.reorder_tensor(self.K_w(K))
        v = self.reorder_tensor(self.V_w(V))
        shaped_mask = torch.repeat_interleave(attention_mask, self.num_heads, 0)
        out = self.attention(q, k, v, shaped_mask) # still (batch size*heads, queries, hidden per head).  
        z = out.reshape( (out.shape[0] // self.num_heads, self.num_heads, -1, self.per_hidden) ) # split back into batch size and heads 
        concatted = z.permute((0, 2, 1, 3)).reshape(z.shape[0], -1, self.total_hidden) # concat heads -> (batch, queries, hidden per head * heads)
        # apply weights
        return self.C_w(concatted) # (batch, queries, input dim)
        

    def reorder_tensor(self, X: torch.Tensor) -> torch.Tensor:
        """Take input of size (batch, n q/k/v, n hidden * n heads) and reshape to (batch, n heads, n q/k/v, n hidden)"""
        intermediate = (
            X
            .reshape(-1, X.shape[1], self.num_heads, self.per_hidden) # (batch size, queries, heads, hidden per head) -> dim 0 grouped by queries
            .permute(0, 2, 1, 3) # (batch size, heads, queries, hidden per head) -> dim 0 grouped by heads
        )
        return (
            intermediate
            .reshape( (intermediate.shape[0]*intermediate.shape[1], -1, self.per_hidden) ) # (batch size*heads, queries, hidden per head) to fit attention
        )

In [121]:
a = torch.tensor([ # batch size 2, 3 queries, len 2
    [ 
        [0., 0.], # q1
        [1., 1.], # q2
        [20., 20.] # q3
    ],
    [ 
        [100., 100.], # q1
        [1000., 1000.], # q2
        [1., 0.] # q3
    ],
])

#### scratch

In [66]:
# 2 heads, 7 hidden
dense = nn.LazyLinear(14, bias=False)
z: torch.Tensor = dense(a) # (1, 3, 1, 14)
#print(z.shape)
intermediate = z.reshape(-1, a.shape[1], 2, 7 # (batch size, queries, heads, hidden per head) -> dim 0 grouped by queries
).permute(0, 2, 1, 3) # (batch size, heads, queries, hidden per head) -> dim 0 grouped by heads
final = intermediate.reshape(4, 3, 7) # combine batch size and num heads for attn
attn = Attention(dropout=0)
final_attn = attn(final, final, final, torch.zeros((4, 3, 3), dtype=torch.bool))
print(final_attn)

tensor([[[ 1.1109e+00, -1.2897e+00,  9.5064e-01, -1.4500e+00,  5.3992e+00,
           2.3769e+00, -2.4162e+00],
         [ 3.1684e+00, -3.6783e+00,  2.7112e+00, -4.1355e+00,  1.5399e+01,
           6.7790e+00, -6.8910e+00],
         [ 3.1741e+00, -3.6850e+00,  2.7161e+00, -4.1430e+00,  1.5426e+01,
           6.7912e+00, -6.9034e+00]],

        [[-3.1309e+00, -2.0866e+00, -5.8265e-02, -9.1937e+00,  2.0690e+00,
           2.0120e+00,  5.7614e-01],
         [-8.9453e+00, -5.9617e+00, -1.6647e-01, -2.6268e+01,  5.9114e+00,
           5.7485e+00,  1.6461e+00],
         [-8.9453e+00, -5.9617e+00, -1.6647e-01, -2.6268e+01,  5.9114e+00,
           5.7485e+00,  1.6461e+00]],

        [[ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
           3.3956e+02, -3.4517e+02],
         [ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
           3.3956e+02, -3.4517e+02],
         [ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
           3.3956e+02, -3

In [67]:

#print(final_attn.shape)
batch = final_attn.reshape((final.shape[0] // 2, 2, -1, 7)) # break heads out from batch size
batch

tensor([[[[ 1.1109e+00, -1.2897e+00,  9.5064e-01, -1.4500e+00,  5.3992e+00,
            2.3769e+00, -2.4162e+00],
          [ 3.1684e+00, -3.6783e+00,  2.7112e+00, -4.1355e+00,  1.5399e+01,
            6.7790e+00, -6.8910e+00],
          [ 3.1741e+00, -3.6850e+00,  2.7161e+00, -4.1430e+00,  1.5426e+01,
            6.7912e+00, -6.9034e+00]],

         [[-3.1309e+00, -2.0866e+00, -5.8265e-02, -9.1937e+00,  2.0690e+00,
            2.0120e+00,  5.7614e-01],
          [-8.9453e+00, -5.9617e+00, -1.6647e-01, -2.6268e+01,  5.9114e+00,
            5.7485e+00,  1.6461e+00],
          [-8.9453e+00, -5.9617e+00, -1.6647e-01, -2.6268e+01,  5.9114e+00,
            5.7485e+00,  1.6461e+00]]],


        [[[ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
            3.3956e+02, -3.4517e+02],
          [ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
            3.3956e+02, -3.4517e+02],
          [ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
      

In [68]:
# double check that if we only put one item in the batch through it comes out the same as before
b = torch.tensor([ # batch size 1, 3 queries, len 2
    [ 
        [0., 0.], # q1
        [1., 1.], # q2
        [20., 20.] # q3
    ],

])

z: torch.Tensor = dense(b)
intermediate_half = z.reshape(-1, a.shape[1], 2, 7
).permute(0, 2, 1, 3) 
final_final = intermediate_half.reshape(2, 3, 7) 
final_attn_half = attn(final_final, final_final, final_final, torch.zeros((2, 3, 3), dtype=torch.bool))
final_attn_half.reshape((final_final.shape[0] // 2, 2, -1, 7)) # break heads out from batch size

tensor([[[[  1.1109,  -1.2897,   0.9506,  -1.4500,   5.3992,   2.3769,  -2.4162],
          [  3.1684,  -3.6783,   2.7112,  -4.1355,  15.3985,   6.7790,  -6.8910],
          [  3.1741,  -3.6850,   2.7161,  -4.1430,  15.4262,   6.7912,  -6.9034]],

         [[ -3.1309,  -2.0866,  -0.0583,  -9.1937,   2.0690,   2.0120,   0.5761],
          [ -8.9453,  -5.9617,  -0.1665, -26.2677,   5.9114,   5.7485,   1.6461],
          [ -8.9453,  -5.9617,  -0.1665, -26.2677,   5.9114,   5.7485,   1.6461]]]],
       grad_fn=<ViewBackward0>)

working as intended! ^

In [91]:
head_concat = batch.permute((0, 2, 1, 3)).reshape(batch.shape[0], 3, -1)
head_concat

tensor([[[ 1.1109e+00, -1.2897e+00,  9.5064e-01, -1.4500e+00,  5.3992e+00,
           2.3769e+00, -2.4162e+00, -3.1309e+00, -2.0866e+00, -5.8265e-02,
          -9.1937e+00,  2.0690e+00,  2.0120e+00,  5.7614e-01],
         [ 3.1684e+00, -3.6783e+00,  2.7112e+00, -4.1355e+00,  1.5399e+01,
           6.7790e+00, -6.8910e+00, -8.9453e+00, -5.9617e+00, -1.6647e-01,
          -2.6268e+01,  5.9114e+00,  5.7485e+00,  1.6461e+00],
         [ 3.1741e+00, -3.6850e+00,  2.7161e+00, -4.1430e+00,  1.5426e+01,
           6.7912e+00, -6.9034e+00, -8.9453e+00, -5.9617e+00, -1.6647e-01,
          -2.6268e+01,  5.9114e+00,  5.7485e+00,  1.6461e+00]],

        [[ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
           3.3956e+02, -3.4517e+02, -4.4726e+02, -2.9809e+02, -8.3235e+00,
          -1.3134e+03,  2.9557e+02,  2.8743e+02,  8.2306e+01],
         [ 1.5871e+02, -1.8425e+02,  1.3581e+02, -2.0715e+02,  7.7131e+02,
           3.3956e+02, -3.4517e+02, -4.4726e+02, -2.9809e+02, -8.3235e+

In [92]:
dense = nn.LazyLinear(1, bias = False)
dense(head_concat)

tensor([[[  2.0134],
         [  5.7465],
         [  5.7527]],

        [[287.6353],
         [287.6353],
         [287.6353]]], grad_fn=<UnsafeViewBackward0>)

#### Testing

In [122]:
mha = MultiHeadAttention(2, 100, 5, 0)

In [123]:
out = mha(a, a, a)

In [124]:
out

tensor([[[   4.2889,    1.0925],
         [   4.0071,   -0.7175],
         [   3.0477,   -2.1254]],

        [[ 149.6081, -110.9588],
         [ 149.6081, -110.9588],
         [ 100.3801, -163.4098]]], grad_fn=<UnsafeViewBackward0>)

## Positional Encoding

## Layer norm

## FFN

## Encoder

## Decoder

## Model

## Data loader